# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Select categorical encoders based on feature meaning, model type, cardinality, and unknown-category risk. 
- Configure one-hot and ordinal encoders to handle infrequent or unseen categories safely. 
- Integrate categorical encoders with column-specific preprocessing in a combined workflow. 


## **1. Choosing Encoders**

### **1.1. Encoder Selection Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_01.jpg?v=1783833902" width="250">



>* Match encoding to category meaning.
>* Avoid false order in nominal labels.

>* Encoder choice depends on model type.
>* Different models prefer different category representations.

>* High-cardinality categories need more careful encoding.
>* Choose encoders robust to unseen categories.



In [ ]:
#@title Python Code - Encoder Selection Basics

# This example compares simple encoder choices.
# It focuses on category meaning basics.
# No machine learning library is used.

import pandas as pd
import numpy as np

# Build a tiny categorical dataset.
df = pd.DataFrame({
    "color": ["red", "blue", "green", "blue"],
    "size": ["small", "medium", "large", "medium"],
})

# Show the original feature meanings.
print("Original categories:")
print(df)

# Integer labels can mislead nominal features.
color_map = {"red": 0, "blue": 1, "green": 2}
df["color_integer"] = df["color"].map(color_map)

# Ordered labels can use meaningful ranks.
size_map = {"small": 1, "medium": 2, "large": 3}
df["size_ordinal"] = df["size"].map(size_map)

# One hot encoding suits nominal categories.
color_dummies = pd.get_dummies(df["color"], prefix="color")

# Combine a compact teaching view.
result = pd.concat([
    df[["color", "color_integer", "size", "size_ordinal"]],
    color_dummies,
], axis=1)

# Print a short explanation summary.
print("\nEncoded view:")
print(result)

# Highlight the selection rule clearly.
print("\nRule:")
print("Use one-hot for names without order.")
print("Use ordinal numbers only for true order.")



### **1.2. Encoder Selection Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_02.jpg?v=1783833905" width="250">



>* Match encoding to category meaning.
>* Preserve order only when it exists.

>* Choose encoding to match model behavior.
>* Avoid false numeric meaning in categories.

>* High-cardinality features increase sparsity and complexity.
>* Choose encoders robust to rare, unseen categories.



### **1.3. Encoder Selection Factors**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_01_03.jpg?v=1783833906" width="250">



>* Match encoding to category meaning and order.
>* Avoid fake numeric order for labels.

>* Model type shapes encoder choice.
>* High-cardinality features need careful tradeoffs.

>* Rare and unseen categories create deployment risk.
>* Choose encoders robust to drift and growth.



## **2. Robust Category Handling**

### **2.1. Unknown Category Handling**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_01.jpg?v=1783833908" width="250">



>* Unseen categories require planned safe handling.
>* One-hot uses fallback; ordinal uses reserved code.

>* Bad unknown handling distorts model meaning.
>* One-hot safely groups unseen categories together.

>* Balance robustness with monitoring and interpretability.
>* Rising unknowns may require model updates.



### **2.2. Unknown Category Safety**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_02.jpg?v=1783833913" width="250">



>* Plan encoders for unseen future categories.
>* Keep transformations stable, valid, and interpretable.

>* Encode unseen categories without breaking transformations.
>* Group rare labels for stability, less detail.

>* Use dedicated unknown values in ordinal encoding.
>* Check model interpretation to avoid false order.



In [ ]:
#@title Python Code - Unknown Category Safety

# This script shows safe category handling.
# It compares unknown category fallback behaviors.
# Results stay small and beginner friendly.

import pandas as pd
import numpy as np

# Build tiny training categories.
train = pd.DataFrame({
    "color": ["red", "blue", "red", "green"]
})

# Build tiny future categories.
test = pd.DataFrame({
    "color": ["red", "yellow", "green", "purple"]
})

# Learn known categories safely.
known = sorted(train["color"].dropna().unique().tolist())

# Create one hot column names.
one_hot_cols = [f"color_{name}" for name in known]

# Encode one hot safely.
def safe_one_hot(series, categories):

    rows = []
    unknown_flags = []

    for value in series:
        row = [int(value == name) for name in categories]
        rows.append(row)
        unknown_flags.append(int(value not in categories))

    frame = pd.DataFrame(rows, columns=one_hot_cols)
    frame["color_unknown"] = unknown_flags
    return frame

# Define meaningful ordinal mapping.
ordinal_map = {"green": 0, "blue": 1, "red": 2}

# Encode ordinal with placeholder.
def safe_ordinal(series, mapping, unknown_value=-1):

    encoded = [mapping.get(value, unknown_value) for value in series]
    return pd.Series(encoded, name="color_ordinal")

# Transform future categories safely.
one_hot_test = safe_one_hot(test["color"], known)
ordinal_test = safe_ordinal(test["color"], ordinal_map)

# Check expected output sizes.
assert len(test) == len(one_hot_test) == len(ordinal_test)

# Summarize safe behavior briefly.
unknown_count = int(one_hot_test["color_unknown"].sum())
print("Known categories:", known)
print("Future values:", test["color"].tolist())
print("Unknown count:", unknown_count)
print("One hot columns:", one_hot_cols + ["color_unknown"])
print("One hot rows:", one_hot_test.values.tolist())
print("Ordinal mapping:", ordinal_map)
print("Ordinal unknown value:", -1)
print("Ordinal rows:", ordinal_test.tolist())



### **2.3. Unknown Category Safety**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_02_03.jpg?v=1783833919" width="250">



>* Unseen categories are normal in changing data.
>* Safe encoding prevents crashes and unpredictable behavior.

>* Plan safe fallback for unseen categories.
>* Group rare values to reduce ambiguity.

>* Reserve a special code for unknowns.
>* Test, monitor, and revise unknown handling.



## **3. Encoded Output Workflows**

### **3.1. Columnwise Encoding Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_01.jpg?v=1783833923" width="250">



>* Encode each categorical column by meaning.
>* Apply chosen transforms consistently across columns.

>* Per-column steps create one organized workflow.
>* Consistent encoding prevents mismatches across stages.

>* Columnwise pipelines improve transparency and maintenance.
>* Stable outputs keep production predictions reliable.



### **3.2. Pipeline Integration**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_02.jpg?v=1783833921" width="250">



>* Fit encoding once, reuse it everywhere.
>* Consistent mappings prevent shifted model inputs.

>* Integrated pipelines prevent leakage during evaluation.
>* They handle changing categories consistently over time.

>* Pipelines unify preprocessing into one reusable workflow.
>* They improve deployment, auditing, and consistent updates.



### **3.3. Column Transformer Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_05/Lecture_A/image_03_03.jpg?v=1783833925" width="250">



>* Different columns get different preprocessing steps.
>* Outputs combine into one consistent workflow.

>* Reuses category mappings for consistent predictions.
>* Prevents column mixups and handles unknowns.

>* Column branches enable modular preprocessing changes.
>* Combined outputs support reliable end-to-end modeling.



# <font color="#418FDE" size="6.5" uppercase>**Categorical Encoding**</font>


In this lecture, you learned to:
- Select categorical encoders based on feature meaning, model type, cardinality, and unknown-category risk. 
- Configure one-hot and ordinal encoders to handle infrequent or unseen categories safely. 
- Integrate categorical encoders with column-specific preprocessing in a combined workflow. 

In the next Lecture (Lecture B), we will go over 'Imputation Targets'